In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType,LongType, ArrayType

In [0]:
redemption_schema = StructType([
    StructField("member_id", StringType(), True),
    StructField("feed_date", StringType(), True),
    StructField("redemptions",
                ArrayType(
                 StructType([
                StructField("txn_id", StringType(), True),
                StructField("txn_date", StringType(), True),
                StructField("partner", StringType(), True),
                StructField("miles_redeemed", LongType(), True),
                StructField("status", StringType(), True)
            ])   
        ),
        True
    )
])

Read the JSON file

In [0]:
JSON_PATH = "/Volumes/workspace/skypoints/raw_files/redemption.json"

redemption_df = (
    spark.read
    .schema(redemption_schema)
    .option("multiLine", "true")
    .option("mode", "PERMISSIVE")
    .json(JSON_PATH)
)

In [0]:
display(redemption_df)

Flatten the redemption array

In [0]:
exploded_df = redemption_df.withColumn(
    "redemption",
    F.explode("redemptions")
)

In [0]:
exploded_df = exploded_df.select(
    F.col("member_id").alias("MEMBER_ID"),
    F.to_date(
        F.try_to_timestamp(
        F.col("feed_date"), F.lit("yyyyMMdd")
    )).alias("FEED_DATE"),
    F.col("redemption.txn_id").alias("TXN_ID"),
    F.to_date(
        F.try_to_timestamp(
            F.col("redemption.txn_date"),
            F.lit("yyyyMMdd")
        )
    ).alias("TXN_DATE"),
    F.col("redemption.partner").alias("PARTNER"),
    F.col("redemption.miles_redeemed").alias("MILES_REDEEMED"),
    F.col("redemption.status").alias("STATUS")

)

In [0]:
display(exploded_df)

In [0]:
exploded_df = exploded_df.withColumn(
    "DQ_ERRORS",
    F.concat_ws(
        " | ",
            F.when(
                F.col("MEMBER_ID").isNull() |
                (F.trim(F.col("MEMBER_ID")) == ""),
                F.lit("MISSING_MEMBER_ID")
                ),
            F.when(
                F.col("FEED_DATE").isNull(),
                F.lit("INVALID_FEED_DATE")
            ),
            F.when(
                F.col("TXN_ID").isNull() |
                (F.trim(F.col("TXN_ID")) == ""),
                F.lit("MISSING_TXN_ID")
            ),

        F.when(
            F.col("TXN_DATE").isNull(),
            F.lit("INVALID_TXN_DATE")
        ),

        F.when(
            F.col("PARTNER").isNull()
            | (F.trim(F.col("PARTNER")) == ""),
            F.lit("MISSING_PARTNER")
        ),

        F.when(
            F.col("MILES_REDEEMED").isNull()
            | (F.col("MILES_REDEEMED") <= 0),
            F.lit("INVALID_MILES_REDEEMED")
        ),

        F.when(
            F.col("STATUS").isNull()
            | (F.trim(F.col("STATUS")) == ""),
            F.lit("MISSING_STATUS")
        )

    )
)

In [0]:
redemption_valid_df = exploded_df.filter(F.col("DQ_ERRORS") == "")

redemption_rejected_df = exploded_df.filter(F.col("DQ_ERRORS") != "")

In [0]:
print("Total transactions:", redemption_df.count())
print("Valid transactions:", redemption_valid_df.count())
print("Rejected transactions:", redemption_rejected_df.count())

display(redemption_rejected_df)

In [0]:
# Check duplicate transaction IDs

display(
    redemption_valid_df
    .groupBy("TXN_ID")
    .count()
    .filter(F.col("count") > 1)
)

Save the redemption data into Delta tables

In [0]:
GOLD_REDEMPTION = "workspace.skypoints.gold_redemption"

REDEMPTION_REJECTS = "workspace.skypoints.silver_redemption_rejects"

In [0]:
(
    redemption_valid_df
    .drop("DQ_ERRORS")
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REDEMPTION)
)

In [0]:
(
    redemption_rejected_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(REDEMPTION_REJECTS)
)

Validate the Gold redemption table

In [0]:
%sql
SELECT *
FROM skypoints.gold_redemption;

In [0]:
%sql
-- Calculate total miles redeemed
SELECT
    MEMBER_ID,
    SUM(MILES_REDEEMED) AS TOTAL_MILES_REDEEMED
FROM skypoints.gold_redemption
GROUP BY MEMBER_ID;